In [ ]:
from datasets import load_dataset
import pandas as pd

Load the validation split in streaming mode
Purpose : To download and load data progressively in small chunks as you iterate through it

In [ ]:
dataset = load_dataset(
    "united-we-care/United-Syn-Med", split="validation", streaming=True
)

first_sample = next(iter(dataset))
print("Available Keys in Dataset Sample:")
for key in first_sample.keys():
  print(f"- {key}")

print("\nSample Data Preview:")
print(first_sample)


In [ ]:
import os
from huggingface_hub import hf_hub_download

# Take the file_name from your dataset sample
sample_file_name = first_sample["file_name"]

# Test 1: Check root path
try:
  path_root = hf_hub_download(
      repo_id="united-we-care/United-Syn-Med",
      filename=sample_file_name,
      repo_type="dataset",
  )
  print(f"Success! Audio found at root path: {sample_file_name}")
except Exception as e:
  print(f"Not at root path: {e}")

# Test 2: Check inside 'data/' folder
try:
  path_data = hf_hub_download(
      repo_id="united-we-care/United-Syn-Med",
      filename=f"data/{sample_file_name}",
      repo_type="dataset",
  )
  print(f"Success! Audio found inside data/ folder: data/{sample_file_name}")
except Exception as e:
  print(f"Not inside data/ folder: {e}")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
# List files in the dataset repository
files = api.list_repo_files(
    repo_id="united-we-care/United-Syn-Med", repo_type="dataset"
)

# Print out top-level files and folders
print("Repository Files Structure:")
for f in files[:30]:  # Print first 30 files/folders
  print(f"- {f}")

In [ ]:
import tarfile
from huggingface_hub import hf_hub_download

# 1. Download the validation audio tarball
tar_local_path = hf_hub_download(
    repo_id="united-we-care/United-Syn-Med",
    filename="data/audio/validation.tar.gz",
    repo_type="dataset",
)

# 2. Open the tar archive and extract the target audio file
target_file_name = "drug-female-9c34a81d-dc2f-4cb9-87ec-7675719ab82d.mp3"

with tarfile.open(tar_local_path, "r:gz") as tar:
  # Inspect member names if needed, or extract directly
  # (Sometimes paths inside the tar include a subfolder prefix)
  matching_members = [
      m for m in tar.getmembers() if target_file_name in m.name
  ]

  if matching_members:
    member = matching_members[0]
    file_obj = tar.extractfile(member)
    audio_bytes = file_obj.read()
    print(
        f"Successfully extracted {target_file_name} ({len(audio_bytes)} bytes)!"
    )
  else:
    print("File not found inside tarball.")